# RAG Example: Loading Your Own Data into ChromaDB

This notebook shows how to load **any text data** into ChromaDB and query it.

We'll use the famous "Wear Sunscreen" speech (Chicago Tribune, 1997) as our example dataset.

Original source: [Salem Public Library — 1999 Yearbook](http://history.salem.lib.oh.us/yearbooks/1999/2facesplits/1999op_Part9.pdf)

## Step 1 — Read and chunk the text

The text file has the speech split into paragraphs (separated by blank lines).
We split on `\n\n` so each paragraph becomes one chunk in our database.

In [ ]:
with open("../data/sunscreen_speech.txt") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"{len(chunks)} chunks loaded\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:80]}...\n")

## Step 2 — Load into ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient("../chroma")

# Delete collection if it already exists (so we can re-run this cell)
try:
    client.delete_collection("sunscreen_speech")
except Exception:
    pass

collection = client.create_collection(name="sunscreen_speech")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Added {collection.count()} chunks to the 'sunscreen_speech' collection")

## Step 3 — Query the collection

ChromaDB uses the query text to find the most semantically similar chunks.
This is the same pattern we use in `5_rag.ipynb` with the nutrition database.

In [ ]:
results = collection.query(
    query_texts=["What does the author say about moving to another place?"],
    n_results=3,
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i + 1}:")
    print(doc)
    print()